# Unidad 12 · Inferencia bayesiana

Reproduce el ejemplo resuelto (la cervecería: botellas con baja carbonatación y volumen de llenado) y la pieza del caso Estampa Sur. Lee los CSV de `../datos/`. Al final, `RES` junta las cifras que el verificador compara.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
DATOS = '../datos/'

## Ejemplo A · Beta-binomial

Priori Beta(3, 57); 9 botellas con baja carbonatación de 120.

In [ ]:
bot = pd.read_csv(DATOS + 'u12_ejemplo_botellas.csv')
x, n = int(bot.baja_carbonatacion.sum()), len(bot)
a, b = 3, 57
post = stats.beta(a + x, b + n - x)
print('posteriori Beta(%d, %d)' % (a + x, b + n - x))
print('media', post.mean(), ' moda', (a + x - 1) / (a + b + n - 2))
print('intervalo de credibilidad 95 %:', post.interval(0.95))
print('P(theta > 8 %) =', post.sf(0.08))

In [ ]:
# priori plana Beta(1, 1), para comparar
plana = stats.beta(1 + x, 1 + n - x)
print(plana.mean(), plana.interval(0.95), plana.sf(0.08))

## Ejemplo B · Normal-normal (sigma conocida)

In [ ]:
vol = pd.read_csv(DATOS + 'u12_ejemplo_llenado.csv').volumen_ml
mu0, s0, sigma = 331.0, 0.8, 1.8
tau0, taud = 1 / s0**2, len(vol) / sigma**2
mun = (tau0 * mu0 + taud * vol.mean()) / (tau0 + taud)
sn = (tau0 + taud) ** -0.5
post_mu = stats.norm(mun, sn)
print('media posterior', mun, ' desviación', sn)
print('intervalo de credibilidad 95 %:', post_mu.interval(0.95))
print('P(mu < 330) =', post_mu.cdf(330))

## Caso · Estampa Sur, tasa de defecto de Algodones Sur

In [ ]:
d = pd.read_csv(DATOS + 'defectos.csv').set_index('proveedor')
k, m = d.loc['Algodones Sur', ['defectuosas', 'revisadas']]
post_as = stats.beta(4 + k, 76 + m - k)
print('posteriori Beta(%d, %d)' % (4 + k, 76 + m - k))
print('media', post_as.mean(), ' IC 95 %', post_as.interval(0.95), ' P(theta > 6 %)', post_as.sf(0.06))

In [ ]:
# ¿Algodones Sur es peor que Telas Norte? Simulación desde las dos posterioris
kt, mt = d.loc['Telas Norte', ['defectuosas', 'revisadas']]
rng = np.random.default_rng(1212)
sim_as = rng.beta(4 + k, 76 + m - k, 400_000)
sim_tn = rng.beta(4 + kt, 76 + mt - kt, 400_000)
p_mayor = (sim_as > sim_tn).mean()
print('P(Algodones Sur > Telas Norte) =', p_mayor)

In [ ]:
RES = {'ej_media': post.mean(), 'ej_ic_inf': post.ppf(0.025), 'ej_ic_sup': post.ppf(0.975), 'ej_p_sobre': post.sf(0.08),
       'ej_mun': mun, 'ej_sn': sn, 'ej_p_bajo': post_mu.cdf(330),
       'caso_media': post_as.mean(), 'caso_ic_inf': post_as.ppf(0.025), 'caso_ic_sup': post_as.ppf(0.975),
       'caso_p_sobre': post_as.sf(0.06), 'caso_p_as_mayor': p_mayor}
RES